In [1]:
import numpy as np


class PauliBasis:
    def __init__(self, vectors, labels, superbasis=None):
        if vectors.shape[1] != vectors.shape[2]:
            raise ValueError(
                "Pauli basis vectors must be square matrices, got shape {}x{}"
                    .format(vectors.shape[1], vectors.shape[2]))

        self.vectors = vectors
        self.labels = labels
        self._superbasis = superbasis

    
    def __eq__(self, other):
        if isinstance(other, PauliBasis):
            return (self.vectors.shape == other.vectors.shape and
                    np.allclose(self.vectors, other.vectors))
        else:
            return False

    def __hash__(self):
        return hash(np.ascontiguousarray(self.vectors).data.tobytes())

    @property
    def dim_hilbert(self):
        return self.vectors.shape[1]

    @property
    def dim_pauli(self):
        return self.vectors.shape[0]

    @property
    def superbasis(self):
        return self._superbasis or self

    def subbasis(self, indices):
        """Return a subbasis of this basis.

        Parameters
        ----------
        indices : list of int
            Indices of basis vectors to include in subbasis

        Returns
        -------
        PauliBasis
        """
        return PauliBasis(self.vectors[list(indices)],
                          [self.labels[i] for i in indices], self)

    def hilbert_to_pauli_vector(self, rho):
        return np.einsum("xab, ba -> x", self.vectors, rho, optimize=True)

    def is_orthonormal(self):
        i = np.einsum("xab, yba -> xy", self.vectors,
                      self.vectors, optimize=True)
        assert np.allclose(i, np.eye(self.dim_pauli))

In [2]:
from itertools import count
from functools import lru_cache

_sqrt2i = np.sqrt(0.5)

@lru_cache(maxsize=64)
def gell_mann(dim_hilbert):
    """A Pauli basis consisting of the generalization of Pauli matrices for
    higher dimensions, the generalized Gell-Mann matrices.

    Gell-Mann matrices are Hermitian and traceless, except the first,
    which is the identity [1]_ [2]_.

    References
    ----------
    .. [1] https://en.wikipedia.org/wiki/Generalizations_of_Pauli_matrices
    .. [2] https://en.wikipedia.org/wiki/Gell-Mann_matrices
    """

    def diagonal(index, zeros):
        if index == 0:
            diag = np.ones(dim_hilbert) / np.sqrt(dim_hilbert)
        else:
            diag = np.zeros(dim_hilbert)
            diag[:index] = 1
            diag[index] = -index
            diag /= np.sqrt(index * (index + 1))

        for i, d in enumerate(diag):
            zeros[i, i] = d

    def off_diagonal(i, j, zeros):
        if i < j:
            zeros[i, j] = _sqrt2i
            zeros[j, i] = _sqrt2i
        else:
            zeros[i, j] = 1j * _sqrt2i
            zeros[j, i] = -1j * _sqrt2i

    vectors = np.zeros((dim_hilbert * dim_hilbert, dim_hilbert, dim_hilbert),
                       dtype=complex)
    # noinspection PyTypeChecker
    labels = np.full(dim_hilbert * dim_hilbert, None, dtype=object)
    counter = count()

    for i in range(dim_hilbert):
        for j in range(dim_hilbert):
            num = next(counter)
            labels[num] = ("γ{}{}".format(i, j))
            if i == j:
                diagonal(i, vectors[num])
            else:
                off_diagonal(i, j, vectors[num])

    return PauliBasis(vectors, labels)

twolevel_ixyz = PauliBasis(
    vectors=_sqrt2i * np.array([[[1, 0], [0, 1]],
                                [[0, 1], [1, 0]],
                                [[0, -1j], [1j, 0]],
                                [[1, 0], [0, -1]]]),
    labels=np.array(("I", "X", "Y", "Z"), dtype=object)
)


In [124]:
from __future__ import annotations
import math
from typing import Iterable

from jax import Array
from jax import numpy as jnp


class OperatorBasis:
    def __init__(self, operators: Array, labels: Iterable[str]) -> None:
        num_dims = len(operators.shape)
        if num_dims != 3:
            raise ValueError(
                f"Operator basis tensors must have shape (pauli_dim, dim, dim), got shape {operators.shape}."
            )
        pauli_dim, dim, other_dim = operators.shape
        if dim != other_dim:
            raise ValueError(
                f"Operator basis tensors must be square matrices, got shape ({dim}, {other_dim})."
            )
        if pauli_dim > dim**2:
            raise ValueError(
                "Number of operators must be less than or equal to the Hilbert space dimension squared."
            )

        num_labels = len(labels)
        if pauli_dim != num_labels:
            raise ValueError(
                f"Number of labels must match the number of operators, got {pauli_dim} operators and {len(labels)} labels."
            )

        self._pauli_dim = pauli_dim
        self._dim = dim

        self._ops = operators
        self._labels = list(labels)

    @property
    def dim(self):
        return self._dim

    @property
    def pauli_dim(self):
        return self._pauli_dim

    def subbasis(self, inds: Iterable[int]) -> OperatorBasis:
        ind_arr = jnp.array(inds)
        sub_ops = jnp.take(self._ops, ind_arr, axis=0)
        sub_labels = [self._labels[ind] for ind in inds]
        return OperatorBasis(sub_ops, sub_labels)

    def expand_dim(self, dim: int) -> OperatorBasis:
        if dim <= self._dim:
            raise ValueError(
                "New dimension must be greater than the current dimension."
            )

        diff = dim - self._dim
        pad_widths = ((0, 0), (0, diff), (0, diff))
        expanded_ops = jnp.pad(self._ops, pad_widths)
        return OperatorBasis(expanded_ops, self._labels)
    
    def truncate_dim(self, dim: int) -> OperatorBasis:
        if dim >= self._dim:
            raise ValueError(
                "New dimension must be less than the current dimension."
            )
        trunc_ops = self._ops[:, :dim, :dim]
        return OperatorBasis(trunc_ops, self._labels)

    def is_orthogonal(self) -> bool:
        inner_prod = jnp.einsum("xij, yij -> xy", self._ops, self._ops)
        diag = jnp.diagonal(inner_prod)
        offdiag_mat = inner_prod - jnp.diag(diag)
        num_elems = int(jnp.count_nonzero(offdiag_mat))
        result = math.isclose(num_elems, 0)
        return result

    def is_orthonormal(self) -> bool:
        inner_prod = jnp.einsum("xij, yji -> xy", self._ops, self._ops)
        identity = jnp.identity(self.pauli_dim)
        result = bool(jnp.allclose(inner_prod, identity))
        return result

    def transform(self, trans_op: Array) -> OperatorBasis:
        transformed_ops = jnp.einsum(
            "ji, ajk, kl -> ail", jnp.conj(trans_op), self._ops, trans_op
        )
        return OperatorBasis(transformed_ops, self._labels)

In [119]:
ops = jnp.array(
    [[[1, 0], [0, 1]],
    [[0, 1], [1, 0]],
    [[0, -1j], [1j, 0]],
    [[1, 0], [0, -1]]]
) / np.sqrt(2)
labels = ("I", "X", "Y", "Z")

basis = OperatorBasis(ops, labels)

In [120]:
basis.is_orthonormal()

True

In [121]:
basis.is_orthogonal()

True

In [123]:
basis.subbasis((0, 1))._labels

['I', 'X']

In [28]:
twolevel_ixyz.computational_basis_vectors

array([[ 0.70710678+0.j,  0.        +0.j,  0.        +0.j,
         0.70710678+0.j],
       [ 0.70710678+0.j,  0.        +0.j,  0.        +0.j,
        -0.70710678+0.j]])

In [37]:
b.trace_index

0

In [42]:
np.einsum("ij->ji", np.array([[0, 1], [-1, 0]]))

array([[ 0, -1],
       [ 1,  0]])

In [72]:
a = jnp.array(np.random.rand(4, 2, 2))
b = jnp.array(np.random.rand(4, 2, 2))
res = jnp.einsum("xab, yba -> xy", a, b)

In [99]:
testo = (1, 2, 3, 4, 5)

In [101]:
inds = (0, 1, 2)
testo[]

TypeError: tuple indices must be integers or slices, not tuple

In [81]:
%%timeit
res = jnp.zeros((4, 3, 3))
res.at[:, :2, :2].set(b)

209 µs ± 2.26 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [82]:
%%timeit
res = jnp.pad(b, ((0, 0), (0, 1), (0, 1)), mode="constant", constant_values=0)

13.1 µs ± 136 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [154]:
import math
from typing import Tuple, Any

from jax import Array
from jax import numpy as jnp


def get_pauli_ops(
    normalize: bool = False, dtype: Any | None = None
) -> Tuple[Array, Array, Array, Array]:
    """
    get_pauli_ops Returns the qubit Pauli operators, more specifically in the order of I, X, Y, Z.

    Returns
    -------
    Tuple[Array]
        The Pauli operators I, X, Y, Z (in that order).
    """
    if dtype is not None:
        try:
            dtype = jnp.dtype(dtype)
        except TypeError as exc:
            raise ValueError(f"Invalid datatype {dtype}.") from exc
    else:
        dtype = jnp.complex64

    prefactor = 1 / math.sqrt(2) if normalize else 1

    pauli_i = prefactor * jnp.array([[1, 0], [0, 1]], dtype=dtype)
    pauli_x = prefactor * jnp.array([[0, 1], [1, 0]], dtype=dtype)
    pauli_y = prefactor * jnp.array([[0, -1j], [1j, 0]], dtype=dtype)
    pauli_z = prefactor * jnp.array([[1, 0], [0, -1]], dtype=dtype)

    # NOTE: could be a dictionary instead, but this seems more useful.
    return pauli_i, pauli_x, pauli_y, pauli_z


In [155]:
pauli_ops = get_pauli_ops(normalize=True)
operators = jnp.stack(pauli_ops)
labels = ("I", "X", "Y", "Z")
pauli_basis = OperatorBasis(operators, labels)

In [156]:
exp_basis = pauli_basis.expand_dim(3)

In [157]:
exp_basis.is_orthonormal()

True

In [158]:
exp_basis._ops

Array([[[ 0.70710677+0.j        ,  0.        +0.j        ,
          0.        +0.j        ],
        [ 0.        +0.j        ,  0.70710677+0.j        ,
          0.        +0.j        ],
        [ 0.        +0.j        ,  0.        +0.j        ,
          0.        +0.j        ]],

       [[ 0.        +0.j        ,  0.70710677+0.j        ,
          0.        +0.j        ],
        [ 0.70710677+0.j        ,  0.        +0.j        ,
          0.        +0.j        ],
        [ 0.        +0.j        ,  0.        +0.j        ,
          0.        +0.j        ]],

       [[ 0.        +0.j        ,  0.        -0.70710677j,
          0.        +0.j        ],
        [ 0.        +0.70710677j,  0.        +0.j        ,
          0.        +0.j        ],
        [ 0.        +0.j        ,  0.        +0.j        ,
          0.        +0.j        ]],

       [[ 0.70710677+0.j        ,  0.        +0.j        ,
          0.        +0.j        ],
        [ 0.        +0.j        , -0.70710677+0.j     

In [153]:
jnp.issubdtype(jnp.complex128, complex)

True